In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
pip install youtube-transcript-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.1/485.1 kB 10.0 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi

def extract_video_id(url: str) -> str:
     """
       Extract the YouTube video ID from a URL.
       Raises ValueError if no 'v' parameter is found.
     """
     parsed = urlparse(url)
     qs = parse_qs(parsed.query)
     video_ids = qs.get('v')

     if not video_ids:
        raise ValueError(f"No video id found in URL: {url}")
     return video_ids[0]

if __name__ == "__main__":
          url = "https://www.youtube.com/watch?v=osKyvYJ3PRM&t=465s"

          video_id = extract_video_id(url)

          api = YouTubeTranscriptApi()
          fetched = api.fetch(video_id, languages=['en']) # returns a FetchedTranscript

          text = "\n".join(snippet.text for snippet in fetched)


In [5]:
print(text)

this video is going to give you
everything you need to go from knowing
absolutely nothing about artificial
intelligence and large language models
to having a solid foundation of how
these revolutionary Technologies work
over the past year artificial
intelligence has completely changed the
world with products like chat PT
potentially appending every single
industry and how people interact with
technology in general and in this video
I will be focusing on llms how they work
ethical cons iterations applications and
so much more and this video was created
in collaboration with an incredible
program called AI camp in which high
school students learn all about
artificial intelligence and I'll talk
more about that later in the video let's
go so first what is an llm is it
different from Ai and how is chat GPT
related to all of this llms stand for
large language models which is a type of
neural network that's trained on massive
amounts of text data it's generally
trained on data that can be fou

In [6]:
def chunk_text(text, max_len=2000, overlap=200):
    """
    text      : the full transcript
    max_len   : max number of characters per chunk
    overlap   : number of overlapping characters to keep context
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_len
        chunk = text[start:end]
        
        # add chunk
        chunks.append(chunk)
        
        # move start, keep some overlap
        start = end - overlap
    
    return chunks

# Example usage
transcript = text  # assume this is a big string variable
chunks = chunk_text(transcript, max_len=2000, overlap=200)
print(f"Number of chunks: {len(chunks)}")


Number of chunks: 15


In [7]:
from huggingface_hub import login
login()

In [8]:
from transformers import pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

2025-12-22 20:38:27.395229: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766435907.605243      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766435907.659260      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766435908.103305      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766435908.103338      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766435908.103341      55 computation_placer.cc:177] computation placer alr

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [9]:
chunk_summaries = []

for i, chunk in enumerate(chunks):
    summary = summarizer(
        chunk,
        max_length=150,
        min_length=40,
        do_sample=False
    )[0]["summary_text"]
    
    chunk_summaries.append(summary)
    print(f"Chunk {i+1} summarized")


Chunk 1 summarized
Chunk 2 summarized
Chunk 3 summarized
Chunk 4 summarized
Chunk 5 summarized
Chunk 6 summarized
Chunk 7 summarized
Chunk 8 summarized
Chunk 9 summarized


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Chunk 10 summarized
Chunk 11 summarized
Chunk 12 summarized
Chunk 13 summarized


Your max_length is set to 150, but your input_length is only 148. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=74)


Chunk 14 summarized
Chunk 15 summarized


In [10]:
combined_summary = " ".join(chunk_summaries)


In [12]:
final_summary = summarizer(
    combined_summary,
    max_length=500,
    min_length=60,
    do_sample=False
)[0]["summary_text"]

print("Final Summary:\n", final_summary)

Final Summary:
 The history of large language models traces all the way back to the first language model from 1966. The first recurrent neural network was created in 1924 or RNN they weren't really able to learn until 1972. In the early 2000s the field of AI evolved very slowly with language models far behind what we see today. This all changed in 2017 when Google released a research paper about a new technology called Transformers.
